# Notebook 2: Bronze Layer - Ingestion des données brutes

**Objectif**: Ingestérer les données depuis S3 vers la couche Bronze du lakehouse.

## Qu'est-ce que la couche Bronze?

La **couche Bronze** est la première couche du lakehouse:
- Elle stocke les données **brutes**, telles que reçues de la source
- Ajoute des **métadonnées techniques** (date d'ingestion, nom du fichier, etc.)
- Garde le format original (PascalCase, types d'origine)
- Est **partitionnée** par date d'ingestion pour les performances

## Flux de données

```
S3 (CSV) ──► Lecture Spark ──► Enrichissement métadonnées ──► Table Iceberg Bronze
```

---
## INITIALISATION

In [ ]:
# Configuration du chemin vers le projet
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Import des modules Spark et configuration
from lakehouse.spark_session import get_spark
from lakehouse.settings import AWS_BUCKET

# Création de la session Spark
spark = get_spark("bronze-ingestion")

# Configuration de la branche Nessie main
spark.conf.set("spark.sql.catalog.nessie.ref", "main")

# Création des namespaces
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.bronze")

print("=" * 60)
print("COUCHE BRONZE - INGESTION DES DONNÉES")
print("=" * 60)

## BATCH 1 - PREMIÈRE INGESTION

**Scénario**: C'est le matin, le premier fichier de ventes arrive.

Fichier: `sales_batch_001.csv`

### 1. Lecture du premier batch depuis S3

In [ ]:
# Import des fonctions PySpark
from pyspark.sql.functions import current_timestamp, current_date, lit

# Chemin du premier batch
batch1_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_001.csv"

# Lecture avec options CSV appropriées
batch1_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch1_path)
)

batch1_count = batch1_raw.count()
print(f"BATCH 1 lu: {batch1_count:,} enregistrements")

### Aperçu des données brutes

In [ ]:
print("=== Aperçu des données brutes ===")
batch1_raw.show(5, truncate=False)

print("\n=== Schéma des données ===")
batch1_raw.printSchema()

### 2. Enrichissement avec métadonnées techniques

La couche Bronze ajoute des colonnes de traçabilité:

| Colonne | Description |
|---------|-------------|
| `ingestion_date` | Date de l'ingestion (partition) |
| `ingestion_ts` | Timestamp exact de l'ingestion |
| `source_file` | Nom du fichier source |
| `source_system` | Système d'origine |
| `batch_id` | Identifiant du batch |

In [ ]:
# Enrichissement avec métadonnées
batch1_bronze = (
    batch1_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_001.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_1"))
)

print("=== Données enrichies (avec métadonnées) ===")
batch1_bronze.select("Order ID", "Order Date", "Sales", "ingestion_date", "ingestion_ts", "batch_id").show(5, truncate=False)

### 3. Suppression de l'ancienne table (si elle existe)

In [ ]:
# Nettoyage: suppression des tables si elles existent (pour une démo propre)
spark.sql("DROP TABLE IF EXISTS nessie.bronze.sales")

print("Tables existantes supprimées")

### 4. Création de la table Iceberg Bronze

**Points clés**:
- Format: **Iceberg** (format de table ACID)
- Partition: par `ingestion_date` (optimise les requêtes par date)
- Catalogue: **Nessie** (versionnement automatique)

In [ ]:
from pyspark.sql.functions import col

# Création de la table Bronze avec partitionnement sur ingestion_date
batch1_bronze.writeTo("nessie.bronze.sales").using("iceberg").partitionedBy(col("ingestion_date")).create()

print(f"BATCH 1 -> Table Bronze créée: {batch1_count:,} enregistrements")

### 5. Vérification de la table Bronze

In [ ]:
bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Nombre d'enregistrements dans Bronze: {bronze_count:,}")

In [ ]:
print("=== Échantillon de la table Bronze ===")
spark.sql("SELECT * FROM nessie.bronze.sales LIMIT 5").show(truncate=False)

### 6. Inspecter les métadonnées Iceberg

Iceberg conserve un historique de toutes les modifications. Chaque écriture crée un nouveau **snapshot**.

In [ ]:
print("=== Historique des snapshots ===")
spark.sql("""
    SELECT 
        made_current_at,
        snapshot_id,
        parent_id
    FROM nessie.bronze.sales.history
    ORDER BY made_current_at
""").show(truncate=False)

## BATCH 2 - DEUXIÈME INGESTION (MODE APPEND)

**Scénario**: C'est le midi, un nouveau fichier de ventes arrive.

**Point clé**: Les nouvelles données sont **ajoutées** aux existantes (mode APPEND).

In [ ]:
# Lecture du deuxième batch depuis S3
batch2_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_002.csv"

batch2_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch2_path)
)

batch2_count = batch2_raw.count()
print(f"BATCH 2 lu: {batch2_count:,} enregistrements")

In [ ]:
# Enrichissement et ajout à la table Bronze (mode APPEND)
batch2_bronze = (
    batch2_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_002.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_2"))
)

# Mode APPEND: les données sont ajoutées aux existantes
batch2_bronze.writeTo("nessie.bronze.sales").append()

new_bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"BATCH 2 -> Ajouté à la table Bronze (APPEND)")
print(f"Nouveau total Bronze: {new_bronze_count:,} (+{new_bronze_count - batch1_count})")

## BATCH 3 - TROISIÈME INGESTION

In [ ]:
# Lecture du troisième batch depuis S3
batch3_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_003.csv"

batch3_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch3_path)
)

batch3_count = batch3_raw.count()
print(f"BATCH 3 lu: {batch3_count:,} enregistrements")

In [ ]:
# Enrichissement et ajout à Bronze
batch3_bronze = (
    batch3_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_003.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_3"))
)

batch3_bronze.writeTo("nessie.bronze.sales").append()

final_bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"BATCH 3 -> Ajouté à Bronze")
print(f"Total Bronze: {final_bronze_count:,}")

## RÉSUMÉ DE L'INGESTION BRONZE

In [ ]:
print("=" * 60)
print("RÉSUMÉ DE L'INGESTION BRONZE")
print("=" * 60 + '\n')

print(f"Total Bronze: {final_bronze_count:,} enregistrements")

print("Distribution par batch:")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("\nHistorique des snapshots:")
spark.sql("""
    SELECT 
        made_current_at,
        snapshot_id,
        parent_id
    FROM nessie.bronze.sales.history
    ORDER BY made_current_at
""").show(truncate=False)

print("Observation: Chaque batch crée un nouveau snapshot!")
print("On peut revenir à n'importe quel moment dans le temps.")

In [ ]:
print("""
╔════════════════════════════════════════════════════════════╗
║         COUCHE BRONZE - INGESTION TERMINÉE                 ║
╠════════════════════════════════════════════════════════════╣
║                                                            ║
║  Table               nessie.bronze.sales                   ║
║  Format              Iceberg (ACID, Time Travel)           ║
║  Partition           ingestion_date                        ║
║  Enregistrements     {:>10,}                               ║
║  Snapshots           {:>10} (capacity de time travel!)     ║
║                                                            ║
║  Métadonnées ajoutées:                                     ║
║    • ingestion_date, ingestion_ts                          ║
║    • source_file, source_system, batch_id                  ║
║                                                            ║
╚════════════════════════════════════════════════════════════╝
"".format(final_bronze_count, spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales.history").first()[0]))

print("→ Prochaine étape: Exécuter '03_silver_transformations.ipynb'")